# 14 · 실시간 작업공간 (분산앵커 웹캠 루프)

노트북 13에서 통합한 분산앵커 파이프라인을 **웹캠 실시간 루프**로 돌린다. 보드 검출 대신 **매 프레임 마커 지도로 로컬라이즈** → DA 검출 → **카메라 합성 | 가상 3D 씬** 나란히.

- 마커 몇 개만 보여도 로컬라이즈되므로 카메라가 움직여도, 근접해도 동작(최소앵커 실험: 2D로 퍼진 4~5개 권장, 1~2개 의존 금지).
- 물체를 올리면 3D에 생기고 치우면 사라짐(매 프레임 재검출) — 최초 목표.
- 가상 3D 씬은 **실제 카메라 포즈 방향**에 맞춰 앞뒤가 일치.
- **주의: 웹캠 K가 필요**(마커지도는 물리치수라 카메라 무관, 하지만 로컬라이즈엔 그 카메라의 내부파라미터 필요). `au.load_intrinsics`가 키규약(camera_matrix/dist_coeffs 또는 K/dist) 자동 처리.

> 이 환경엔 웹캠이 없어 아래 1)에서 **씬 사진 1장을 한 프레임처럼** 넣어 실시간 루프가 보여줄 화면을 검증한다. 실제 루프는 2)의 `run_live_workspace`.
> 검출 방식(top-hat 국소대비)의 상세·검증은 노트북 15 참고.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, glob
import cv2, numpy as np
import matplotlib.pyplot as plt
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
import aruco_utils as au, workspace as ws, scene3d as s3, depth_volume as dv, live_da as ld

OUTPUT_DIR = os.path.join(ROOT, 'output')
SCENE_DIR  = os.path.join(ROOT, 'data', 'scene_images', '31_marker')

corners_map, markers_list, REF, L = ws.load_marker_map(os.path.join(OUTPUT_DIR, 'marker_map.json'))
x0, y0, x1, y1 = ws.map_extent_m(corners_map)
PLANE = (x0, y0, x1, y1)
ORIGIN = (x0*1000, y0*1000)                       # 가상씬 원점 오프셋(id0 X음수 보정)
WS_SZ  = ((x1-x0)*1000+60, (y1-y0)*1000+60)
print(f'작업공간 {(x1-x0)*1000:.0f} x {(y1-y0)*1000:.0f} mm, 마커 {len(corners_map)}개')
pipe = dv.load_depth_model()

## 1) 한 프레임 검증 (웹캠 없이 씬 사진으로)

`process_frame_workspace`(로컬라이즈+DA) → `render_virtual_scene`(origin 오프셋, 카메라 포즈 방향)으로 실시간 루프와 동일한 **카메라 | 가상3D** 뷰를 생성.

> 씬 사진은 폰이라 폰 K 사용. 실제 웹캠 루프는 웹캠 K.

In [ ]:
Kp, distp = au.load_intrinsics(os.path.join(OUTPUT_DIR, 'phone_intrinsics.npz'))   # 씬사진=폰
frame = cv2.imread(sorted(glob.glob(os.path.join(SCENE_DIR, '*.jpg')))[-1])

vis, objs, ok, n, err, pose = ld.process_frame_workspace(
    frame, pipe, Kp, distp, corners_map, markers_list, PLANE, min_area_px=1200)
print(f'로컬라이즈 ok={ok} {n}마커 {err:.1f}px | 물체 {len(objs)}개')
for o in objs:
    c = o['cyl'][0]; print(f"  {o['type']:5s} ({c[0]:+.0f},{c[1]:+.0f})mm {o['label']}")

scene = s3.render_virtual_scene(objs, markers=markers_list, ws=WS_SZ, origin_mm=ORIGIN,
                                title='workspace 3D', cam_pose=pose, plane_xyxy=PLANE)
camh = cv2.resize(vis, (int(vis.shape[1]*scene.shape[0]/vis.shape[0]), scene.shape[0]))
view = np.hstack([camh, scene])
plt.figure(figsize=(15, 5)); plt.imshow(cv2.cvtColor(view, cv2.COLOR_BGR2RGB)); plt.axis('off')
plt.title('live view (simulated from one photo): camera | virtual 3D'); plt.show()

## 2) 웹캠 실시간 루프 (웹캠 연결 시)

`run_live_workspace`: 매 프레임 마커지도 로컬라이즈 → DA 검출(top-hat) → 카메라 | 가상3D. `[s]`스냅 `[q]`종료.

```python
Kw, distw = au.load_intrinsics(os.path.join(OUTPUT_DIR, 'camera_intrinsics.npz'))   # 그 웹캠 K
ld.run_live_workspace(Kw, distw, corners_map, markers_list, PLANE, pipe=pipe,
                      cam_index=0, calib_wh=(1920, 1080), snapshot_dir=OUTPUT_DIR, min_area_px=1200)
```

**셋업 체크리스트**
1. 웹캠을 작업공간 위 근접/탑다운으로 고정(근접샷 조건 = 정확). 원거리 전체 담기는 작은 물체 놓침.
2. 프레임에 지도 마커가 2D로 퍼져 **4~5개** 보이게(1~2개 의존 금지).
3. 그 웹캠을 ChArUco로 1회 캘리브(`01_charuco_calibration`)해 `camera_intrinsics.npz` 준비(해상도도 그 웹캠에 맞게 `calib_wh`).
4. 성능: DA 매 프레임이라 GTX1660 기준 ~2fps. 부드럽게 원하면 하이브리드(11) 방식으로 확장 가능.

In [ ]:
# 웹캠 있으면 주석 해제:
# Kw, distw = au.load_intrinsics(os.path.join(OUTPUT_DIR, 'camera_intrinsics.npz'))
# ld.run_live_workspace(Kw, distw, corners_map, markers_list, PLANE, pipe=pipe,
#                       cam_index=0, calib_wh=(1920, 1080), snapshot_dir=OUTPUT_DIR, min_area_px=1200)
print('웹캠 연결 후 위 셀 주석 해제 실행')

## 정리

- 분산앵커 파이프라인의 **웹캠 실시간 루프 완성**(`ld.run_live_workspace`, `ld.process_frame_workspace`). 보드 없이 넓은 작업공간에서 매 프레임 로컬라이즈+검출+3D 배치.
- 가상 3D 씬은 `origin_mm`로 id0 기준(음수) 좌표를 격자에 정렬, `cam_pose`로 실제 카메라 방향에 맞춤.
- 내부파라미터 로드는 `au.load_intrinsics`로 통일(camera_matrix/dist_coeffs, K/dist 두 규약 자동).
- 한 프레임 검증: 립밤이 카메라·가상3D 양쪽에 올바른 위치로 표시됨.
- 정확도/한계는 13·15와 동일(근접·탑다운=정확, 원거리 작은물체 놓침, 바닥오차 6~10mm=지도/캘리브 한계).
- 다음(선택): 하이브리드(async) 부드러운 실시간, 물체 ID 추적으로 add/remove 안정화, 다중시점 지름·부피, 로봇팔 핸드-아이 캘리브.